# Package Quickstart

This notebook uses the public package API directly.

The basic SILVA equilibrium has the form

$$
z^\star = \sigma\{S_\theta(x)+L_\theta(z^\star)+G_\theta(z^\star)\}.
$$

The forward pass solves for $z^\star$, then a PyTorch head maps the state to
outputs.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/jseluis/silva-networks.git"

def find_local_silva_root():
    candidates = [
        Path.cwd(),
        Path("/content/silva-networks"),
        Path("/content/drive/MyDrive/silva-networks"),
    ]
    root = Path.cwd()
    while root != root.parent:
        candidates.append(root)
        root = root.parent
    for candidate in candidates:
        if (candidate / "src" / "silva_networks").exists():
            return candidate
    return None

root = find_local_silva_root()
if root is not None:
    sys.path.insert(0, str(root / "src"))
elif IN_COLAB and importlib.util.find_spec("silva_networks") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", f"git+{REPO_URL}"])
    root = Path.cwd()
else:
    root = Path.cwd()

In [ ]:
import torch
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 300, "savefig.dpi": 300})
from silva_networks import SILVAGraphNetwork, SolverConfig, resolve_device

torch.manual_seed(0)
device = resolve_device("auto")
device

Create a small graph. The edge tensor has shape $(2,E)$: the first row stores
sources and the second row stores destinations.

In [ ]:
x = torch.randn(10, 5, device=device)
edge_index = torch.tensor(
    [list(range(9)), list(range(1, 10))],
    dtype=torch.long,
    device=device,
)

In [ ]:
model = SILVAGraphNetwork(
    in_dim=5,
    hidden_dims=[16, 16],
    out_dim=3,
    task="node",
    local="graph",
    global_term="mean",
    config=SolverConfig(solver="anderson", max_iter=8, alpha=0.5, history=4),
).to(device)

result = model(x, edge_index=edge_index, return_results=True)
logits = result.output
logits.shape, [round(r.residual, 6) for r in result.solver_results]

The model is an ordinary PyTorch module. Parameters are visible to optimizers,
and gradients flow through the unrolled fixed-point solve used in the public
implementation.

In [ ]:
y = torch.randint(0, 3, (x.shape[0],), device=device)
loss = torch.nn.functional.cross_entropy(logits, y)
loss.backward()
sum(p.grad is not None for p in model.parameters()), float(loss.detach().cpu())

The residual curve records

$$
\|f_\theta(z_k,x)-z_k\|_2
$$

at each solver step. The curve is a practical diagnostic for whether the
chosen damping and solver budget are reasonable on this example.

In [ ]:
plt.figure(figsize=(5, 3))
for layer_index, solver_result in enumerate(result.solver_results):
    plt.plot(solver_result.residuals, marker="o", label=f"layer {layer_index}")
plt.yscale("log")
plt.xlabel("solver step")
plt.ylabel("residual")
plt.legend()
plt.tight_layout()

## Citation and Sources

If this notebook or package is used, cite the software repository:

```text
Dr. Jose Luis Silva. SILVA Networks. Version 1.0.0. MIT License.
https://github.com/jseluis/silva-networks
```

When the work is connected to the SILVA Networks paper, cite the paper as well:

```text
Jose Luis Lima de Jesus Silva. SILVA Networks as Structured Implicit Layers and
Vector Attractors via Dynamic Interaction Fields. 2026. arXiv:2607.28989.
https://arxiv.org/abs/2607.28989
```

Background references used in the tutorial suite include:

- Deep Equilibrium Models, Bai, Kolter, and Koltun, NeurIPS 2019:
  https://arxiv.org/abs/1909.01377
- Multiscale Deep Equilibrium Models, Bai, Koltun, and Kolter, NeurIPS 2020:
  https://arxiv.org/abs/2006.08656
- Stabilizing Equilibrium Models by Jacobian Regularization, Bai, Koltun, and
  Kolter, ICML 2021: https://arxiv.org/abs/2106.14342
- Graph Attention Networks, Velickovic et al., ICLR 2018:
  https://arxiv.org/abs/1710.10903
- Attention Is All You Need, Vaswani et al., 2017:
  https://arxiv.org/abs/1706.03762